# Teks — Named Entity Recognition (NER) dengan CRF
**Kapan dipakai:** tiap kata (token) dalam kalimat diberi tag entitas, misalnya nama orang,
tempat, organisasi, dengan format BIO (`B-PER` = awal nama orang, `I-PER` = lanjutan,
`O` = bukan entitas).

**Langkah:** kelompokkan token per kalimat → feature tiap token (kata itu, awalan/akhiran,
kata sebelum/sesudah) → CRF → evaluasi per token.

**Data demo:** NERP (kalimat Bahasa Indonesia, tag BIO). Test berlabel dipakai sebagai
evaluasi akhir (jangan dipakai untuk tuning).

In [1]:
import pandas as pd
import sklearn_crfsuite
from sklearn_crfsuite import metrics

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/text/ner"   # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/nerp_train.csv"
TEST_CSV = f"{DATA_FOLDER}/nerp_test.csv"

SENTENCE_COL = "sentence_id"
TOKEN_COL = "token"
TAG_COL = "tag"

## Baca data
Satu baris = satu token. `keep_default_na=False` supaya token seperti "NA" tidak jadi kosong.

In [3]:
train = pd.read_csv(TRAIN_CSV, keep_default_na=False)
test = pd.read_csv(TEST_CSV, keep_default_na=False)
print("train:", train.shape, "| test:", test.shape)
print("test terurut per kalimat:", test[SENTENCE_COL].is_monotonic_increasing)
train.head(10)

train: (57630, 3) | test: (14001, 3)
test terurut per kalimat: True


,sentence_id,token,tag
0,0,budiyanto,O
1,0,juga,O
2,0,mengimbau,O
3,0,masyarakat,O
4,0,yang,O
5,0,datang,O
6,0,agar,O
7,0,tetap,O
8,0,tertib,O
9,0,dan,O


## EDA: jumlah tiap tag dan panjang kalimat

In [4]:
print(train[TAG_COL].value_counts())
sentence_lengths = train.groupby(SENTENCE_COL).size()
print(sentence_lengths.describe())

tag
O        49308
B-PPL     1760
B-PLC     1612
B-IND     1118
I-PPL     1032
I-IND      758
I-PLC      729
I-EVT      555
B-EVT      392
B-FNB      239
I-FNB      127
Name: count, dtype: int64
count    2500.000000
mean       23.052000
std        10.549626
min         1.000000
25%        16.000000
50%        21.000000
75%        28.000000
max       106.000000
dtype: float64


## Kelompokkan token per kalimat
CRF butuh daftar kalimat; tiap kalimat = daftar token (dan daftar tag).

In [5]:
def to_sentences(table, col):
    sentences = []
    for sentence_id, group in table.groupby(SENTENCE_COL):
        sentences.append(list(group[col]))
    return sentences


train_tokens = to_sentences(train, TOKEN_COL)
train_tags = to_sentences(train, TAG_COL)
test_tokens = to_sentences(test, TOKEN_COL)
test_tags = to_sentences(test, TAG_COL)
print(len(train_tokens), "kalimat train,", len(test_tokens), "kalimat test")
print(train_tokens[0])
print(train_tags[0])

2500

 kalimat train, 600 kalimat test
['budiyanto', 'juga', 'mengimbau', 'masyarakat', 'yang', 'datang', 'agar', 'tetap', 'tertib', 'dan', 'tidak', 'merusak', 'fasilitas-fasilitas', 'publik', 'yang', 'ada', 'serta', 'tidak', 'berbuat', 'anarkistis', 'selama', 'acara', 'berlangsung', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## Feature per token
Tambah/hapus baris di fungsi ini untuk mencoba feature lain.

In [6]:
def token_features(sentence, i):
    word = sentence[i]
    features = {"bias": 1.0, "kata": word.lower(), "huruf_besar": word.istitle()}
    features["angka"] = word.isdigit()
    features["akhiran3"] = word[-3:].lower()
    features["awalan3"] = word[:3].lower()
    if i > 0:
        features["kata_sebelum"] = sentence[i - 1].lower()
    else:
        features["awal_kalimat"] = True
    if i < len(sentence) - 1:
        features["kata_sesudah"] = sentence[i + 1].lower()
    else:
        features["akhir_kalimat"] = True
    return features


def sentence_features(sentence):
    features = []
    for i in range(len(sentence)):
        features.append(token_features(sentence, i))
    return features


print(token_features(train_tokens[0], 0))

{'bias': 1.0, 'kata': 'budiyanto', 'huruf_besar': False, 'angka': False, 'akhiran3': 'nto', 'awalan3': 'bud', 'awal_kalimat': True, 'kata_sesudah': 'juga'}


## Ubah semua kalimat menjadi daftar feature

In [7]:
X_train = []
for sentence in train_tokens:
    X_train.append(sentence_features(sentence))
X_test = []
for sentence in test_tokens:
    X_test.append(sentence_features(sentence))

## Model (CRF)
`c1`, `c2` = kekuatan regularisasi. Pilih SATU model: hapus `#` di baris yang ingin dipakai.

In [8]:
model = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100,
                             all_possible_transitions=True)
# model = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.01, max_iterations=200,
#                              all_possible_transitions=True)
# model = sklearn_crfsuite.CRF(algorithm="l2sgd", c2=0.1, max_iterations=200)
# model = sklearn_crfsuite.CRF(algorithm="ap", max_iterations=50)   # averaged perceptron
# model = sklearn_crfsuite.CRF(algorithm="pa", max_iterations=50)   # passive aggressive

model.fit(X_train, train_tags)

,algorithm,'lbfgs'
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,100
,min_freq,None
,all_possible_states,None
,num_memories,None
,epsilon,None
,period,None
,delta,None


## Evaluasi di test
Skor per token, tanpa tag `O` (tag `O` sangat banyak dan membuat skor terlihat bagus palsu).

In [9]:
test_predictions = model.predict(X_test)
entity_tags = sorted(set(train[TAG_COL]) - {"O"})
weighted_f1 = metrics.flat_f1_score(test_tags, test_predictions, average="weighted",
                                    labels=entity_tags)
print("F1 weighted:", round(weighted_f1, 4))
print(metrics.flat_classification_report(test_tags, test_predictions, labels=entity_tags,
                                         digits=3))

F1 weighted: 0.6299
              precision    recall  f1-score   support

       B-EVT      0.554     0.432     0.485        95
       B-FNB      0.737     0.418     0.533        67
       B-IND      0.838     0.533     0.652       272
       B-PLC      0.806     0.688     0.742       362
       B-PPL      0.829     0.599     0.695       461
       I-EVT      0.270     0.367     0.311       128
       I-FNB      0.846     0.306     0.449        36
       I-IND      0.596     0.458     0.518       203
       I-PLC      0.664     0.515     0.580       165
       I-PPL      0.774     0.658     0.711       266

   micro avg      0.708     0.560     0.625      2055
   macro avg      0.691     0.497     0.568      2055
weighted avg      0.733     0.560     0.630      2055



## Submission
Model sudah dilatih di seluruh train. Tebakan per kalimat disambung menjadi satu daftar per
token; urutannya sama dengan file test karena kalimat sudah terurut.

In [10]:
predicted_tags = []
for sentence_tags in test_predictions:
    predicted_tags.extend(sentence_tags)

submission = pd.DataFrame()
submission["sentence_id"] = test["sentence_id"]
submission["token"] = test["token"]
submission["tag"] = predicted_tags
submission.to_csv("submission.csv", index=False)
submission.head()

,sentence_id,token,tag
0,0,"""",O
1,0,demi,O
2,0,kebersamaan,O
3,0,",",O
4,0,karena,O
